In [55]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,Time,V1,V2,V3,V4,V5,V6,V7,V8,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,1,143352.0,1.955041,-0.380783,-0.315013,0.330155,-0.509374,-0.086197,-0.627978,0.035994,...,0.238197,0.968305,0.053208,-0.278602,-0.044999,-0.216780,0.045168,-0.047145,9.99,0
1,2,117173.0,-0.400975,-0.626943,1.555339,-2.017772,-0.107769,0.168310,0.017959,-0.401619,...,-0.153485,0.421703,0.113442,-1.004095,-1.176695,0.361924,-0.370469,-0.144792,45.90,0
2,3,149565.0,0.072509,0.820566,-0.561351,-0.709897,1.080399,-0.359429,0.787858,0.117276,...,-0.314638,-0.872959,0.083391,0.148178,-0.431459,0.119690,0.206395,0.070288,11.99,0
3,4,93670.0,-0.535045,1.014587,1.750679,2.769390,0.500089,1.002270,0.847902,-0.081323,...,0.063525,0.443431,-0.072754,0.448192,-0.655203,-0.181038,-0.093013,-0.064931,117.44,0
4,5,82655.0,-4.026938,1.897371,-0.429786,-0.029571,-0.855751,-0.480406,-0.435632,1.313760,...,-0.480691,-0.230369,0.250717,0.066399,0.470787,0.245335,0.286904,-0.322672,25.76,0


In [56]:
subtask1 = len(df_train[(df_train.Class == 1) & (df_train.Amount > df_train.Amount.mean())])

from scipy.spatial.distance import mahalanobis
import numpy as np

features = [col for col in df_test.columns if col != 'id' and col != 'Time']
df_2 = df_train[df_train.Class == 1][features]
mean_vec = df_2.mean().values
cov_matrix = np.cov(df_2.values, rowvar=False)
inv_cov_matrix = np.linalg.inv(cov_matrix)
distances = [mahalanobis(row, mean_vec, inv_cov_matrix) for row in df_2.values]
subtask2 = round(np.mean(distances), 2)

df_subtasks = pd.DataFrame({
    "subtaskID": [1, 2],
    "datapointID": [1, 1],
    "answer": [subtask1, subtask2]
})

df_subtasks

,subtaskID,datapointID,answer
0,1,1,127.0
1,2,1,5.1


In [57]:
X = df_train[features]
y = df_train.Class

X_final = df_test[features]

X.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,1.955041,-0.380783,-0.315013,0.330155,-0.509374,-0.086197,-0.627978,0.035994,1.054560,-0.030441,...,-0.125390,0.238197,0.968305,0.053208,-0.278602,-0.044999,-0.216780,0.045168,-0.047145,9.99
1,-0.400975,-0.626943,1.555339,-2.017772,-0.107769,0.168310,0.017959,-0.401619,0.040378,0.611115,...,-0.470372,-0.153485,0.421703,0.113442,-1.004095,-1.176695,0.361924,-0.370469,-0.144792,45.90
2,0.072509,0.820566,-0.561351,-0.709897,1.080399,-0.359429,0.787858,0.117276,-0.131275,-0.638222,...,0.012227,-0.314638,-0.872959,0.083391,0.148178,-0.431459,0.119690,0.206395,0.070288,11.99
3,-0.535045,1.014587,1.750679,2.769390,0.500089,1.002270,0.847902,-0.081323,0.371579,0.560595,...,-0.253757,0.063525,0.443431,-0.072754,0.448192,-0.655203,-0.181038,-0.093013,-0.064931,117.44
4,-4.026938,1.897371,-0.429786,-0.029571,-0.855751,-0.480406,-0.435632,1.313760,0.536044,1.221746,...,-0.012320,-0.480691,-0.230369,0.250717,0.066399,0.470787,0.245335,0.286904,-0.322672,25.76


In [58]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.svm import LinearSVC
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

model = CatBoostClassifier(silent=True)

cross_val_score(model, X, y, cv=3, scoring='f1').mean()

0.8592663643762353

In [59]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [60]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.id,
    "answer": y_preds
})

df_preds

,subtaskID,datapointID,answer
0,3,227846,1
1,3,227847,0
2,3,227848,0
3,3,227849,0
4,3,227850,0
...,...,...,...
56957,3,284803,0
56958,3,284804,0
56959,3,284805,0
56960,3,284806,0


In [61]:
df_preds.answer.value_counts()

answer
0    56883
1       79
Name: count, dtype: int64

In [62]:
submission = pd.concat([df_subtasks, df_preds])
submission

,subtaskID,datapointID,answer
0,1,1,127.0
1,2,1,5.1
0,3,227846,1.0
1,3,227847,0.0
2,3,227848,0.0
...,...,...,...
56957,3,284803,0.0
56958,3,284804,0.0
56959,3,284805,0.0
56960,3,284806,0.0


In [63]:
submission.to_csv("submission.csv", index=False)